# 06 Learned methods on the test cohort (H2, H3)
Every trained network is applied to the clinical-grade protocols of the test subjects, then tracked and scored exactly like the classical methods.

In [ ]:
DEV_MODE = True    # True: small test run (dev: block of config.yaml). False: full experiment (full: block).

import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from retinopath.core import load_config, get_device, seed_all, Timer
from retinopath import pipeline as PL

cfg = load_config(DEV_MODE); device = get_device(); seed_all(cfg.seed)
RES = cfg.P.out / "results"; FIG = cfg.P.out / "figures"; RES.mkdir(exist_ok=True); FIG.mkdir(exist_ok=True)
print(f"DEV_MODE={DEV_MODE} | device={device} | outputs -> {cfg.P.out}")
if DEV_MODE: print("DEV_MODE: small test run. Its outputs are kept apart in outputs/dev and are not experiment results.")

In [ ]:
import copy, torch
from retinopath.train import load_trained, net_fodf, trained_runs
subjects = PL.make_subject_lists(cfg); test = [PL.session(s) for s in subjects["test"]]
clin = [p for p in cfg.clinical_protocols if p in cfg.run_protocols]; runs = trained_runs(cfg); print("trained runs:", runs)
from tqdm.auto import tqdm
with Timer(cfg, "06_inference"):
    for v, s in runs:
        c = copy.deepcopy(cfg); model, meta = load_trained(c, v, s, device)
        for ses in tqdm(test, desc=f"{v}_s{s}"):
            for p in clin: net_fodf(c, ses, p, f"net_{v}_s{s}", model, device)

In [ ]:
with Timer(cfg, "06_tracking_scores"):
    df = PL.run_grid(cfg, test, [(p, f"net_{v}_s{s}") for p in clin for v, s in runs], "learned")
df["variant"] = df.method.str.extract(r"net_(.*)_s\d+")[0]
df.groupby(["protocol", "variant"])[["n_pairs", "rsi_ree_all", "acc", "peak_err_deg", "wdice_ref"]].median().round(3)